# SPINE-GPE v7 — PNADc Certifier v1.1.0

Certificação dos suplementos diretos de plataformas digitais da PNADc em **2022T4** e **2024T3**.

Regra central:

```python
platform_delivery_direct = (SD14001 == 1) & (S140093 == 1)
```

O notebook preserva `S140093 == 1` separadamente como uso bruto de aplicativo de entrega.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
ROOT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("Existe:", ROOT.exists())

In [ ]:
# Localiza o script em /content ou na pasta persistente scripts/.
SCRIPT_NAME = "SPINE_GPEv7_PNADC_CERTIFIER_v1.1.0.py"
SCRIPT_CANDIDATES = [
    Path("/content") / SCRIPT_NAME,
    ROOT / "scripts" / SCRIPT_NAME,
    ROOT / SCRIPT_NAME,
]
SCRIPT = next((p for p in SCRIPT_CANDIDATES if p.exists()), None)

if SCRIPT is None:
    raise FileNotFoundError(
        "Envie SPINE_GPEv7_PNADC_CERTIFIER_v1.1.0.py para /content "
        "ou coloque-o em SPINE-GPEv7/scripts/."
    )
print("SCRIPT:", SCRIPT)

In [ ]:
# Instala somente dependências ausentes. Não depende de arquivo requirements.
import importlib
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.2",
    "pyarrow": "pyarrow>=17",
    "scipy": "scipy>=1.12",
    "requests": "requests>=2.31",
    "urllib3": "urllib3>=2.2",
    "bs4": "beautifulsoup4>=4.12",
    "openpyxl": "openpyxl>=3.1",
    "xlrd": "xlrd>=2.0",
    "lxml": "lxml>=5",
}

missing = []
for module, package in PACKAGES.items():
    try:
        imported = importlib.import_module(module)
        version = getattr(imported, "__version__", "disponível")
        print(f"OK       {module}: {version}")
    except ImportError:
        print(f"FALTANDO {module} -> {package}")
        missing.append(package)

if missing:
    result = subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--prefer-binary",
            "--retries", "10", "--timeout", "120",
            *missing,
        ],
        text=True,
        capture_output=True,
        check=False,
    )
    print(result.stdout[-12000:])
    print(result.stderr[-12000:])
    if result.returncode != 0:
        raise RuntimeError(f"pip falhou com exit code {result.returncode}")
else:
    print("Todas as dependências já estão disponíveis.")

In [ ]:
# Validação sintática do script.
import py_compile
py_compile.compile(str(SCRIPT), doraise=True)
print("py_compile: OK")

In [ ]:
# Confirma os microdados e o lock da Fase 0.
import json

DATA_PNADC = ROOT / "data_pnadc"
for expected in ("PNADC_042022.txt", "PNADC_032024.txt"):
    matches = list(DATA_PNADC.rglob(expected))
    print(expected, "->", matches[:3])

PHASE0_LOCK = ROOT / "00_admin" / "PHASE0_LOCK.json"
print("PHASE0_LOCK:", PHASE0_LOCK)
if not PHASE0_LOCK.exists():
    raise FileNotFoundError(PHASE0_LOCK)
phase0 = json.loads(PHASE0_LOCK.read_text(encoding="utf-8"))
print("Fase 0:", phase0.get("status"))
if phase0.get("status") != "RELEASED":
    raise RuntimeError("A Fase 0 precisa estar RELEASED.")

In [ ]:
# Auditoria rápida: fontes, layouts, largura fixed-width e variáveis críticas.
audit_cmd = [
    sys.executable, str(SCRIPT),
    "--root", str(ROOT),
    "--mode", "audit",
    "--strict",
]
print(" ".join(audit_cmd))
audit = subprocess.run(audit_cmd, text=True, capture_output=True, check=False)
print(audit.stdout[-16000:])
print(audit.stderr[-16000:])
print("Audit exit code:", audit.returncode)
if audit.returncode != 0:
    raise RuntimeError("Auditoria bloqueada. Examine o log acima.")

In [ ]:
# Certificação completa. Não use --skip-sidra.
certify_cmd = [
    sys.executable, str(SCRIPT),
    "--root", str(ROOT),
    "--mode", "certify",
    "--chunk-rows", "20000",
    "--strict",
]
print(" ".join(certify_cmd))
certify = subprocess.run(certify_cmd, text=True, capture_output=True, check=False)
print(certify.stdout[-24000:])
print(certify.stderr[-24000:])
print("Certification exit code:", certify.returncode)

In [ ]:
# Abre o lock e o relatório mais recente.
LOCK = ROOT / "00_admin" / "PNADC_CERTIFICATION_LOCK.json"
if not LOCK.exists():
    raise FileNotFoundError(LOCK)
lock = json.loads(LOCK.read_text(encoding="utf-8"))
print(json.dumps(lock, ensure_ascii=False, indent=2)[:30000])

REPORT = ROOT / "06_reports" / "pnadc_certification" / "pnadc_certification_report_LATEST.md"
if REPORT.exists():
    report_text = REPORT.read_text(encoding="utf-8", errors="replace")
    print("\n--- RELATÓRIO FINAL ---\n")
    print(report_text[-30000:])

status = lock.get("status")
if status == "BLOCKED":
    raise RuntimeError("Certificação bloqueada; use critical_failures para a próxima correção.")
print("STATUS:", status)

In [ ]:
# Golden check independente da regra de entrega nos Parquets produzidos.
import pandas as pd

BASE = ROOT / "03_processed" / "10_pnadc_certified"
for year in (2022, 2024):
    path = BASE / f"certified_pnadc_platform_{year}.parquet"
    df = pd.read_parquet(
        path,
        columns=[
            "SD14001", "S140093", "delivery_app_use_raw",
            "platform_delivery_direct", "delivery_app_use_nonplatform",
            "survey_weight",
        ],
    )
    w = pd.to_numeric(df["survey_weight"], errors="coerce").fillna(0)
    raw = float(w[df["delivery_app_use_raw"].fillna(False)].sum())
    official = float(w[df["platform_delivery_direct"].fillna(False)].sum())
    nonplatform = float(w[df["delivery_app_use_nonplatform"].fillna(False)].sum())
    print(f"\n{year}")
    print(f"Uso bruto S140093=1: {raw:,.0f}")
    print(f"Entrega oficial SD14001=1 & S140093=1: {official:,.0f}")
    print(f"Uso não plataformizado: {nonplatform:,.0f}")
    print(f"Erro de partição: {raw - official - nonplatform:,.8f}")